In [ ]:
FAMILY_MAP = {
    "Gaussian": "GaussianCopula",
    "Bernoulli": "BernoulliCopula",
    "NegBin": "NegBinCopula",
    "NegBinIRLS": "NegBinIRLSCopula",
    "Poisson": "PoissonCopula",
    "pNMF": "PositiveNMF",
    "ZeroInflatedNegBin": "ZeroInflatedNegBinCopula",
    "ZeroInflatedPoisson": "ZeroInflatedPoissonCopula"
}

In [ ]:
import importlib
import random

def make_formulas(formulas: dict) -> dict:
    formula_strs = {}

    # loop over model parameters
    for param, variables in formulas.items():
        terms = []

        # loop over predictors for that parameter
        for var in variables:
            if "transform" in var and var["transform"].get("type") == "spline":
                terms.append(f"bs({var['variable']}, df={var['transform']['df']})")
            else:
                terms.append(var['variable'])
        formula_strs[f"{param}_formula"] = "~ " + " + ".join(terms) if terms else "~ 1"

    return formula_strs

def generate_data(state):
    # prepare the formula and simulator type
    class_name = FAMILY_MAP[state["family"]]
    simulator_class = getattr(importlib.import_module("scdesigner.simulators"), class_name)
    fmla = make_formulas(state["formulas"])

    # fit the simulator
    simulator = simulator_class(**fmla)
    simulator.fit(state["template"], max_epochs=25)

    # sample new data
    n_cells = state["n_cells"] or len(state["template"])
    ix = random.choices(range(len(state["template"])), k=n_cells)
    new_data = simulator.sample(state["template"].obs.iloc[ix, :])

    return simulator.parameters, new_data

Here is a version of a negative binomial model where we use cell_type as a predictor and don't have any change as a function of pseudotime.

In [ ]:
from scdesigner.datasets import pancreas as example_sce

formulas = {
    "mean": [{"variable": "cell_type"}],
    "dispersion": [{"variable": "cell_type"}],
    "copula": []
}

state = {"template": example_sce, "family": "NegBin", "formulas": formulas, "n_cells": None}
parameters, adata_sim = generate_data(state)

Here are the coefficients for each gene's negative binomial model.

In [ ]:
parameters["marginal"]["mean"]

Here's the estimated copula covariance matrix.

In [ ]:
parameters["copula"]["Intercept"]

We can let the copula correlations depend on the cell type.

In [ ]:
formulas["copula"] = [{"variable": "cell_type"}]
state["formulas"] = formulas
parameters, _ = generate_data(state)

# covariance matrix for this cell type
parameters["copula"]["cell_type[T.Beta]"]

We can allow the means and dispersions to vary smoothly over pseudotime by using spline transformations in the predictors. The `make_formulas` function transforms the object below into strings like `~ bs(pseudotime, df=8)`. I'm breaking that string down into this structure because I think it might map more easily onto user inputs (e.g., selecting variables from a list populated by the `obs` field of the anndata object, then clicking whether a spline should be used). But internally we just need the formula strings -- let me know if there's a more natural data structure from a UI perspective.

In [ ]:
formulas = {
    "mean": [{"variable": "pseudotime", "transform": {"type": "spline", "df": 8}}],
    "dispersion": [{"variable": "pseudotime", "transform": {"type": "spline", "df": 3}}],
    "copula": [{"variable": "cell_type"}]
}

state = {"template": example_sce, "family": "NegBin", "formulas": formulas, "n_cells": None}
generate_data(state)

Here is an example where we use a Gaussian on log transformed data.

In [ ]:
import numpy as np
from copy import deepcopy

formulas = {
    "mean": [{"variable": "pseudotime", "transform": {"type": "spline", "df": 8}}],
    "sdev": [{"variable": "pseudotime", "transform": {"type": "spline", "df": 3}}],
    "copula": [{"variable": "cell_type"}]
}

example_sce_transform = deepcopy(example_sce)
example_sce_transform.X = np.log1p(example_sce_transform.X)
state = {"template": example_sce_transform, "family": "Gaussian", "formulas": formulas, "n_cells": 2000}
generate_data(state)